In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import PathPatch
import matplotlib
matplotlib.rcParams['mathtext.fontset'] = 'stix'
matplotlib.rcParams['font.family'] = 'STIXGeneral'
#matplotlib.pyplot.title(r'ABC123 vs $\mathrm{ABC123}^{123}$')
import scipy as sp
import scipy.stats as stats
import numpy as np

In [ ]:
# box PLOT PARAMETERS AND COLORS
# FIGURE SIZE
font_scale=1.4
figsize_x=5
figsize_y=6
# box COLORS
# Light colors for the box
box_palette = {'W':'#FFE994', 'N3':'#9BDDF9'}
# Dark colors for the dots
swarmplot_palette = {'W':'#FF6600', 'N3':'#2A7FFF'}
# Color for the mean line
meanline_palette = {'W':'#ff8000', 'N3':'#764fe3'} #{'W':'#4727cd', 'N3':'#FF5733'}

In [ ]:
def plot_boxes(ax, metric, NSIM, 
                 box_palette, swarmplot_palette, meanline_palette, font_scale, 
                 legends, xgrid,
                 saveplot = 0):
    file = 'DSET_1_NSIM_' + NSIM + '_'+ metric +'_sub.csv'
    data = pd.read_csv(file)

    # create seaborn context
    sns.set_context('notebook', font_scale=font_scale)

    # Plot the box
    medianprops = dict(linestyle=None, linewidth=3)
    sns.boxplot(y="value",
                x="cond",
                data=data,
                linewidth=1.5,
                hue="cond",
                palette=box_palette,
                saturation=1.0,
                showmeans=True,
                meanline=True,
                legend=False,
                meanprops=dict(linestyle='dashed', color='gray', linewidth=2),
                medianprops=dict(linestyle=None, linewidth=2),
                ax=ax)

    # Plot the swarmplot on top 
    sns.swarmplot(y="value",
                  x="cond",
                  data=data,
                  color="auto",
                  s=8,  # Circle size
                  palette=swarmplot_palette,
                  hue="cond",
                  legend=False,
                  ax=ax)

    # Calculate means #############################
    means = data.groupby("cond")["value"].mean()
    means = means.iloc[::-1]
    datax0W = data[data['cond'] == 'W']
    datax0N = data[data['cond'] == 'N3']
    dataxW = datax0W[["value"]].to_numpy()
    dataxN = datax0N[["value"]].to_numpy()
    datax = [dataxW, dataxN]

    if legends:
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(loc='lower left')

    # Calculate statistical significance between 'W' and 'N3' groups ###############
    stat_test = stats.ranksums(data[data['cond'] == 'W']['value'], data[data['cond'] == 'N3']['value'])
    # Calculate significance level (adjust p-value if necessary)
    p_value = stat_test.pvalue
    if p_value < 0.00001:
        significance_asterisks = '*****'
    elif p_value < 0.0001:
        significance_asterisks = '****'
    elif p_value < 0.001:
        significance_asterisks = '***'
    elif p_value < 0.01:
        significance_asterisks = '**'
    elif p_value < 0.05:
        significance_asterisks = '*'
    else:
        significance_asterisks = '(n.s.)'

    # Add a bar or bracket between the box plots
    (miny, maxy) = ax.get_ylim()
    yposition = maxy
    maxy = maxy + 0.1 * (maxy - miny)
    ax.set_ylim(miny, maxy)
    endwidth = (maxy - miny) / 100
    ax.plot([0, 1], [yposition, yposition], color='black', lw=1.5, zorder=20)  # Adjust the coordinates and style as needed
    ax.plot([0, 0], [yposition - endwidth, yposition + endwidth], color='black', lw=1.5, zorder=20)
    ax.plot([1, 1], [yposition - endwidth, yposition + endwidth], color='black', lw=1.5, zorder=20)
    # Add significance annotation to the plot
    if p_value < 0.05:
        ax.annotate(f'{significance_asterisks}', xy=(0.5, yposition + endwidth), ha='center', fontsize=12)
    else:
        ax.annotate(f'p = {p_value:.5f} {significance_asterisks}', xy=(0.5, yposition + endwidth), ha='center', fontsize=12)
    ################################################################################

    # Change axis labels, ticks, and title
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Wakefulness", "Deep Sleep"])
    ax.set_xlabel('')
    if metric in ['Mi', 'xMi']:
        ax.set_ylabel('Integral Violation of FDT (arbitrary units)')
        ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
    if metric in ['Md', 'xMd']:
        ax.set_ylabel('Differential Violation of FDT (arbitrary units)')
        ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
    if metric in ['MX', 'xMX']:
        ax.set_ylabel('Fluctuation Dissipation Ratio (arbitrary units)')

    # Add horizontal grid
    if xgrid:
        ax.grid(axis='y')
        ax.set_axisbelow(True)

    if saveplot == 1:
        # Save plots
        plt.savefig('fig_'+metric+'_box_NSIM_'+NSIM+'.pdf')
        plt.savefig('fig_'+metric+'_box_NSIM_'+NSIM+'.svg')
        plt.savefig('fig_'+metric+'_box_NSIM_'+NSIM+'.png')
        plt.show()



In [ ]:
NSIM = "10000"
fig, ax = plt.subplots(figsize=(5, 5))

metric = 'Mi'
plot_boxes(ax=ax, metric=metric, NSIM=NSIM, 
           box_palette=box_palette, swarmplot_palette=swarmplot_palette, meanline_palette=meanline_palette, font_scale=font_scale, 
           legends=False, xgrid=False,
           saveplot = 1)

plt.tight_layout()
plt.show()


In [ ]:
NSIM = "10000"
fig, axes = plt.subplots(1, 2, figsize=(8, 5))

metric = 'Md'
plot_boxes(ax=axes[0], metric=metric, NSIM=NSIM, 
           box_palette=box_palette, swarmplot_palette=swarmplot_palette, meanline_palette=meanline_palette, font_scale=font_scale, 
           legends=False, xgrid=False,
           saveplot = 0)

metric = 'Mi'
plot_boxes(ax=axes[1], metric=metric, NSIM=NSIM, 
           box_palette=box_palette, swarmplot_palette=swarmplot_palette, meanline_palette=meanline_palette, font_scale=font_scale, 
           legends=False, xgrid=False,
           saveplot = 0)

plt.tight_layout()
# Save plots
plt.savefig('BOXPLOTS_Md_Mi.pdf')
plt.savefig('BOXPLOTS_Md_Mi.svg')
plt.show()



In [ ]:
NSIM = "10000"
fig, axes = plt.subplots(1, 2, figsize=(8, 5))

metric = 'xMd'
plot_boxes(ax=axes[0], metric=metric, NSIM=NSIM, 
           box_palette=box_palette, swarmplot_palette=swarmplot_palette, meanline_palette=meanline_palette, font_scale=font_scale, 
           legends=False, xgrid=False,
           saveplot = 0)

metric = 'xMi'
plot_boxes(ax=axes[1], metric=metric, NSIM=NSIM, 
           box_palette=box_palette, swarmplot_palette=swarmplot_palette, meanline_palette=meanline_palette, font_scale=font_scale, 
           legends=False, xgrid=False,
           saveplot = 0)

plt.tight_layout()
# Save plots
plt.savefig('BOXPLOTS_xMd_xMi.pdf')
plt.savefig('BOXPLOTS_xMd_xMi.svg')
plt.show()


In [ ]:
### Number of simulations
# NSIMULATIONS = [100, 250, 750, 1000, 2500, 7500, 10000]
NSIMULATIONS = [10000]
METRICS = ['Mi', 'xMi', 'Md', 'xMd', 'MX', 'xMX']

In [ ]:
for NSIM in NSIMULATIONS:
    NSIM = str(NSIM)
    print('#### Number of simulations: ',NSIM)
    for i in range(len(METRICS)):
        metric = METRICS[i]
        fig, ax = plt.subplots(figsize=(5, 5))
        print('Metric: ',metric)
        plot_boxes(ax=ax, metric=metric, NSIM=NSIM, 
                     box_palette=box_palette, swarmplot_palette=swarmplot_palette, meanline_palette=meanline_palette, font_scale=font_scale, 
                     legends=False, xgrid=False)
        plt.tight_layout()
        plt.show()

In [ ]:
def calculate_means_medians(data):
    # Calculate means
    means = data.groupby("cond")["value"].mean()
    means = means.iloc[::-1]
    # Calculate medians
    medians = data.groupby("cond")["value"].median()
    medians = medians.iloc[::-1]

    # Filter data for 'W' and 'N3' conditions
    datax0W = data[data['cond'] == 'W']
    datax0N = data[data['cond'] == 'N3']
    
    # Extract 'value' column and convert to NumPy arrays
    dataxW = datax0W[["value"]].to_numpy()
    dataxN = datax0N[["value"]].to_numpy()
    
    return means, medians

In [ ]:
NSIMULATIONS = [100, 250, 750, 1000, 2500, 7500, 10000]

Mi_W_NSIM, Mi_N_NSIM, Md_W_NSIM, Md_N_NSIM, MX_W_NSIM, MX_N_NSIM  = ([] for i in range(6))
Mi_W_NSIMmed, Mi_N_NSIMmed, Md_W_NSIMmed, Md_N_NSIMmed, MX_W_NSIMmed, MX_N_NSIMmed  = ([] for i in range(6))
xMi_W_NSIM, xMi_N_NSIM, xMd_W_NSIM, xMd_N_NSIM, xMX_W_NSIM, xMX_N_NSIM = ([] for i in range(6))
xMi_W_NSIMmed, xMi_N_NSIMmed, xMd_W_NSIMmed, xMd_N_NSIMmed, xMX_W_NSIMmed, xMX_N_NSIMmed  = ([] for i in range(6))

for s in NSIMULATIONS:
    filex = str('DSET_1_NSIM_%d_Mi_sub.csv' % s)
    data = pd.read_csv(filex)
    means, medians = calculate_means_medians(data)
    Mi_W_NSIM.append(means.W)
    Mi_N_NSIM.append(means.N3)
    Mi_W_NSIMmed.append(medians.W)
    Mi_N_NSIMmed.append(medians.N3)

    filex = str('DSET_1_NSIM_%d_Md_sub.csv' % s)
    data = pd.read_csv(filex)
    means, medians = calculate_means_medians(data)
    Md_W_NSIM.append(means.W)
    Md_N_NSIM.append(means.N3)
    Md_W_NSIMmed.append(medians.W)
    Md_N_NSIMmed.append(medians.N3)

    filex = str('DSET_1_NSIM_%d_MX_sub.csv' % s)
    data = pd.read_csv(filex)
    means, medians = calculate_means_medians(data)
    MX_W_NSIM.append(means.W)
    MX_N_NSIM.append(means.N3)
    MX_W_NSIMmed.append(medians.W)
    MX_N_NSIMmed.append(medians.N3)

    filex = str('DSET_1_NSIM_%d_xMi_sub.csv' % s)
    data = pd.read_csv(filex)
    means, medians = calculate_means_medians(data)
    xMi_W_NSIM.append(means.W)
    xMi_N_NSIM.append(means.N3)
    xMi_W_NSIMmed.append(medians.W)
    xMi_N_NSIMmed.append(medians.N3)

    filex = str('DSET_1_NSIM_%d_xMd_sub.csv' % s)
    data = pd.read_csv(filex)
    means, medians = calculate_means_medians(data)
    xMd_W_NSIM.append(means.W)
    xMd_N_NSIM.append(means.N3)
    xMd_W_NSIMmed.append(medians.W)
    xMd_N_NSIMmed.append(medians.N3)

    filex = str('DSET_1_NSIM_%d_xMX_sub.csv' % s)
    data = pd.read_csv(filex)
    means, medians = calculate_means_medians(data)
    xMX_W_NSIM.append(means.W)
    xMX_N_NSIM.append(means.N3)
    xMX_W_NSIMmed.append(medians.W)
    xMX_N_NSIMmed.append(medians.N3)


## Mi
plt.title(r"$M_I$ convergence")
plt.plot(NSIMULATIONS,Mi_W_NSIM, 'o-', label='W (mean)')
plt.plot(NSIMULATIONS,Mi_W_NSIMmed, 'o-', label='W (median)')
plt.plot(NSIMULATIONS,Mi_N_NSIM, 'o-', label='N3 (mean)')
plt.plot(NSIMULATIONS,Mi_N_NSIMmed, 'o-', label='N3 (median)')
plt.ylabel(r'Integral Violation of FDT')
plt.xlabel(r'Simulations')
plt.xscale("log")
plt.minorticks_on()
plt.legend()
plt.ticklabel_format(style='sci', axis='y', scilimits=(0,0))
plt.tight_layout()
# Save plots
plt.savefig('fig_MI_convergence.pdf')
plt.savefig('fig_MI_convergence.svg')
plt.savefig('fig_MI_convergence.png')
plt.show()

## Md
plt.title(r"$M_V$ convergence")
plt.plot(NSIMULATIONS,Md_W_NSIM, 'o-', label='W (mean)')
plt.plot(NSIMULATIONS,Md_W_NSIMmed, 'o-', label='W (median)')
plt.plot(NSIMULATIONS,Md_N_NSIM, 'o-', label='N3 (mean)')
plt.plot(NSIMULATIONS,Md_N_NSIMmed, 'o-', label='N3 (median)')
plt.ylabel(r'Differential Violation of FDT')
plt.xlabel(r'Simulations')
plt.xscale("log")
plt.minorticks_on()
plt.legend()
plt.ticklabel_format(style='sci', axis='y', scilimits=(0,0))
plt.tight_layout()
# Save plots
plt.savefig('fig_MV_convergence.pdf')
plt.savefig('fig_MV_convergence.svg')
plt.savefig('fig_MV_convergence.png')
plt.show()

## MX
plt.title(r"$M_X$ convergence")
plt.plot(NSIMULATIONS,MX_W_NSIM, 'o-', label='W (mean)')
plt.plot(NSIMULATIONS,MX_W_NSIMmed, 'o-', label='W (median)')
plt.plot(NSIMULATIONS,MX_N_NSIM, 'o-', label='N3 (mean)')
plt.plot(NSIMULATIONS,MX_N_NSIMmed, 'o-', label='N3 (median)')
plt.ylabel(r'Fluctuation Dissipation Ratio')
plt.xlabel(r'Simulations')
plt.xscale("log")
plt.minorticks_on()
plt.legend()
plt.tight_layout()
# Save plots
plt.savefig('fig_MX_convergence.pdf')
plt.savefig('fig_MX_convergence.svg')
plt.savefig('fig_MX_convergence.png')
plt.show()

## xMi
plt.plot(NSIMULATIONS,xMi_W_NSIM, 'o-', label='xMi W (mean)')
plt.plot(NSIMULATIONS,xMi_N_NSIM, 'o-', label='xMi N3 (mean)')
plt.plot(NSIMULATIONS,xMi_W_NSIMmed, 'o-', label='Mi W (median)')
plt.plot(NSIMULATIONS,xMi_N_NSIMmed, 'o-', label='Mi N3 (median)')
plt.ylabel(r'Integral Violation of FDT')
plt.xlabel(r'Simulations')
plt.xscale("log")
plt.minorticks_on()
plt.legend()
plt.show()
## xMd
plt.plot(NSIMULATIONS,xMd_W_NSIM, 'o-', label='xMd W (mean)')
plt.plot(NSIMULATIONS,xMd_N_NSIM, 'o-', label='xMd N3 (mean)')
plt.plot(NSIMULATIONS,xMd_W_NSIMmed, 'o-', label='Mi W (median)')
plt.plot(NSIMULATIONS,xMd_N_NSIMmed, 'o-', label='Mi N3 (median)')
plt.ylabel(r'Differential Violation of FDT')
plt.xlabel(r'Simulations')
plt.xscale("log")
plt.minorticks_on()
plt.legend()
plt.show()
## xMX
plt.plot(NSIMULATIONS,xMX_W_NSIM, 'o-', label='xMX W (mean)')
plt.plot(NSIMULATIONS,xMX_N_NSIM, 'o-', label='xMX N3 (mean)')
plt.plot(NSIMULATIONS,xMX_W_NSIMmed, 'o-', label='Mi med W (median)')
plt.plot(NSIMULATIONS,xMX_N_NSIMmed, 'o-', label='Mi med N3 (median)')
plt.ylabel(r'Fluctuation Dissipation Ratio')
plt.xlabel(r'Simulations')
plt.xscale("log")
plt.minorticks_on()
plt.legend()
plt.show()